🗂️ [STRUCTURE]
**Session 3 — Motion Sensors: The Unlabelled Signals**
**Responsible:** Timo
**Target audience:** Beginner Biomedical Engineering Bachelor students (no Python background).
**Language:** English
**Platform:** JupyterLite (no accounts, runs in the browser)
**Principle:** The notebook is the instrument. Students only edit values inside boxes marked `YOU MAY CHANGE THIS`.

**Cell map (one independent cell per block):**
| Cell | Block | Student action |
|---|---|---|
| 0 | Setup (helpers) | Run only, never edit |
| 1 | IMU in the hand | Run, move the sensor, watch 6 signals |
| 2 | IMU on the foot | Run, perform the foot movements |
| 3 | Mystery recording | Run, record a secret movement |
| 4 | Export data and labels | Fill in labels, run, download CSVs |

**Open points / REVIEW list:**
- REVIEW: Hardware connection (Bluetooth / Capture2Go / Movella DOT) is not implemented. Everything runs in REPLAY MODE (simulated data) until it is finalized.
- REVIEW: Sampling frequency is provisional (`SAMPLING_RATE_HZ = 30`). Confirm the real value of the device.
- REVIEW: Only 4 foot movements are specified in the brief. The 5th is still open (placeholder in Cell 2).
- REVIEW: Signal order is FIXED (Signal 1 to Signal 6) and must be identical in Session 4. Internal mapping (hidden from students) must be confirmed with the real device channel order.
- REVIEW: Align the CSV format with Session 4 (column names, `time_s`, decimal separator, units).
- REVIEW: Where students upload their files (group folder) is undefined.
- REVIEW: Check that `await asyncio.sleep` and live plot refresh work in the target JupyterLite kernel (Pyodide). `time.sleep` would freeze the page.
- REVIEW: Prepare the predefined LLM prompts for the in-browser assistant (e.g. "Explain what this cell does", "Help me change the time window").

---

📝 [TEXT]
## Welcome to Session 3: The Unlabelled Signals

Today you will see what a motion sensor "sees".

Inside the small box you hold there is an **IMU** (Inertial Measurement Unit). It measures movement in 6 different ways at the same time. In the plots, you will only see **Signal 1 to Signal 6**. Nobody tells you which one is which.

**Your mission:**
1. Move the sensor in different ways and watch which signals react.
2. Decide which signal belongs to which sensor and which axis.
3. Record a mystery movement for another group.
4. Save your data and your guesses. You will need them in Session 4.

**How this notebook works:** You do not need to write code. Run each cell with the ▶ button (or `Shift + Enter`). Only change values inside the boxes marked `YOU MAY CHANGE THIS`.

---

🗂️ [STRUCTURE]
**Cell 0: Setup (helpers).** Students run it once at the start. Contains all the code they never edit.

📝 [TEXT]
### Cell 0: Setup
Run this cell once before anything else. It prepares everything for today. You do not need to read or change the code.

In [ ]:
# ==========================================
# 0. SETUP (RUN ONCE, DO NOT CHANGE)
# ==========================================
# What it does: Loads the tools and helper functions used in all other cells.
# Typical error: If a later cell says "name ... is not defined", run this cell again.

import asyncio
import csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# REVIEW: Provisional sampling frequency. Replace with the real device value.
SAMPLING_RATE_HZ = 30

# REVIEW: Fixed signal order. Must match Session 4 and the real device channel order.
# Hidden mapping (do NOT show to students):
#   signal_1..3 = accelerometer X, Y, Z    signal_4..6 = gyroscope X, Y, Z
NUMBER_OF_SIGNALS = 6

# Storage for everything recorded in this session
RECORDINGS = {}


def read_imu_samples(data_source, start_time, n_samples, seed=0):
    """Returns an array of shape (n_samples, 6) with the 6 signals."""
    if data_source == "replay":
        # REPLAY MODE: simulated data, not from a real sensor.
        rng = np.random.default_rng(seed)
        t = start_time + np.arange(n_samples) / SAMPLING_RATE_HZ
        base = np.column_stack([
            np.sin(1.0 * t), np.cos(1.3 * t), 0.5 * np.sin(0.7 * t) + 1.0,
            1.5 * np.cos(0.9 * t), np.sin(2.0 * t), 0.8 * np.cos(1.7 * t),
        ])
        return base + 0.05 * rng.standard_normal(base.shape)
    elif data_source == "live":
        # REVIEW: Real hardware connection goes here (Bluetooth / Capture2Go / Movella DOT).
        raise NotImplementedError("Live sensor not connected yet. Use DATA_SOURCE = 'replay'.")
    else:
        raise ValueError("DATA_SOURCE must be 'replay' or 'live'.")


async def live_plot(data_source, time_window_seconds, duration_seconds, seed=0):
    """Plots the 6 signals while they arrive. No names, no units on purpose."""
    chunk = max(1, SAMPLING_RATE_HZ // 5)          # update about 5 times per second
    window_samples = int(time_window_seconds * SAMPLING_RATE_HZ)
    times, values = [], []
    n_done = 0
    total = int(duration_seconds * SAMPLING_RATE_HZ)
    while n_done < total:
        new = read_imu_samples(data_source, n_done / SAMPLING_RATE_HZ, chunk, seed + n_done)
        times.extend((n_done + np.arange(chunk)) / SAMPLING_RATE_HZ)
        values.extend(new)
        n_done += chunk
        t_arr = np.array(times[-window_samples:])
        v_arr = np.array(values[-window_samples:])
        clear_output(wait=True)
        fig, axes = plt.subplots(NUMBER_OF_SIGNALS, 1, figsize=(10, 8), sharex=True)
        for i, ax in enumerate(axes):
            ax.plot(t_arr, v_arr[:, i])
            ax.set_ylabel(f"Signal {i + 1}")
            ax.set_yticks([])                      # no values: this is a guessing game
        axes[-1].set_xlabel("Time")
        plt.show()
        await asyncio.sleep(1 / 5)   # REVIEW: confirm this refresh works in JupyterLite
    return np.array(times), np.array(values)


def record(data_source, duration_seconds, seed=0):
    """Records without plotting. Returns time and 6 signals."""
    n = int(duration_seconds * SAMPLING_RATE_HZ)
    values = read_imu_samples(data_source, 0, n, seed)
    times = np.arange(n) / SAMPLING_RATE_HZ
    return times, values


print("Setup complete. You can continue with Cell 1.")

---

🗂️ [STRUCTURE]
**Block 1: IMU in the Hand (Labeling Game).** Students hold the sensor and learn which signal reacts to which movement.

📝 [TEXT]
### Cell 1: IMU in the Hand
Hold the sensor in your hand and run this cell. Then move it slowly in different ways:
- Lift it up and down.
- Move it left and right.
- Tilt it forward and backward.
- Rotate it around like a steering wheel.

Watch the 6 signals. Which signals react to which movement? Write down your guesses on paper. You will need them at the end.

In [ ]:
# ==========================================
# 1. IMU IN THE HAND: LIVE GRAPH
# ==========================================
# What it does: Shows 6 unlabelled signals from the IMU while you move it.
# Typical error: "NotImplementedError" means DATA_SOURCE is set to 'live' but no sensor
# is connected yet. Set it back to 'replay'.

# ------------------------------------------
# YOU MAY CHANGE THIS
# ------------------------------------------
DATA_SOURCE = "replay"        # "replay" = simulated data, "live" = real sensor
TIME_WINDOW_SECONDS = 10      # How many seconds are visible on the screen at once
DURATION_SECONDS = 30         # How long the live graph runs
# ------------------------------------------

# REVIEW: With real hardware, DATA_SOURCE = "live" should connect via Bluetooth
# (Capture2Go / Movella DOT). Connection code goes into read_imu_samples() in Cell 0.

await live_plot(DATA_SOURCE, TIME_WINDOW_SECONDS, DURATION_SECONDS, seed=1)
print("Live graph finished. Run the cell again to try more movements.")

---

🗂️ [STRUCTURE]
**Block 2: IMU on the Foot.** Students attach the IMU to the foot and perform fixed movements.
- REVIEW: Only 4 movements are defined in the brief. The 5th movement is missing. Placeholders below must be replaced with the final list.
- Order of movements should be fixed so that Session 4 can reuse the recordings.

📝 [TEXT]
### Cell 2: IMU on the Foot
Now attach the sensor to your foot (follow the photo on the instruction sheet).

You will perform the following movements, one after the other, with a short pause (about 2 seconds of standing still) between them:

1. REVIEW: Movement 1 (to be defined)
2. REVIEW: Movement 2 (to be defined)
3. REVIEW: Movement 3 (to be defined)
4. REVIEW: Movement 4 (to be defined)
5. REVIEW: Movement 5 (not yet specified in the brief)

Run the cell, then perform the movements while the graph is running. Compare the signals with what you saw in Cell 1. Did the same signals react?

In [ ]:
# ==========================================
# 2. IMU ON THE FOOT: RECORD AND PLOT
# ==========================================
# What it does: Records your foot movements and shows the 6 signals live.
# Typical error: If the graph is too short for all movements, increase DURATION_SECONDS.

# ------------------------------------------
# YOU MAY CHANGE THIS
# ------------------------------------------
DATA_SOURCE = "replay"        # "replay" = simulated data, "live" = real sensor
TIME_WINDOW_SECONDS = 10      # How many seconds are visible on the screen at once
DURATION_SECONDS = 40         # Total recording time (enough for all movements)
# ------------------------------------------

# REVIEW: Hardware recording trigger goes here for DATA_SOURCE = "live".

times, values = await live_plot(DATA_SOURCE, TIME_WINDOW_SECONDS, DURATION_SECONDS, seed=2)
RECORDINGS["foot"] = (times, values)
print("Foot recording saved.")

---

🗂️ [STRUCTURE]
**Block 3: The Mystery Recording.** Each group records one movement of its own choice for another group to guess. Matching the guess is done on paper or in the group chat (no code).

📝 [TEXT]
### Cell 3: The Mystery Recording
Choose a movement and **keep it secret**. Run this cell and perform your movement for the whole duration. The graph is not shown during the recording on purpose.

Afterwards, another group will look at your signals and try to guess what you did.

In [ ]:
# ==========================================
# 3. MYSTERY RECORDING
# ==========================================
# What it does: Records a secret movement without showing it, then shows the 6 signals once.
# Typical error: Keep MYSTERY_DURATION between 10 and 20 seconds.

# ------------------------------------------
# YOU MAY CHANGE THIS
# ------------------------------------------
DATA_SOURCE = "replay"        # "replay" = simulated data, "live" = real sensor
MYSTERY_DURATION = 15         # Seconds, between 10 and 20
# ------------------------------------------

# REVIEW: Hardware recording trigger goes here for DATA_SOURCE = "live".

print(f"Secret recording started for {MYSTERY_DURATION} seconds. Move now!")
times, values = record(DATA_SOURCE, MYSTERY_DURATION, seed=3)
RECORDINGS["mystery"] = (times, values)
print("Mystery recording saved. Now show the graph below to the other group.")

fig, axes = plt.subplots(NUMBER_OF_SIGNALS, 1, figsize=(10, 8), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(times, values[:, i])
    ax.set_ylabel(f"Signal {i + 1}")
    ax.set_yticks([])
axes[-1].set_xlabel("Time")
plt.show()

---

🗂️ [STRUCTURE]
**Block 4: Exporting Data and Labels.** Bridge to Session 4. Produces three kinds of files per group:
- `<GROUP_NAME>_foot.csv` and `<GROUP_NAME>_mystery.csv` (data)
- `labels_<GROUP_NAME>.csv` (the group's guessed labels, separate file)

📝 [TEXT]
### Cell 4: Export Your Data and Labels
Now decide which sensor and axis you think each signal is. Use these options **exactly as written**:

`Accel X`, `Accel Y`, `Accel Z`, `Gyro X`, `Gyro Y`, `Gyro Z`, or `Unknown`

Fill them in, enter your group name (**no spaces**), and run the cell. Then download the files and keep them safe. You will need them in Session 4.

**How to download:** In the file browser on the left, right-click each file and choose **Download**.

In [ ]:
# ==========================================
# 4. EXPORT DATA AND LABELS
# ==========================================
# What it does: Saves your recordings and your guessed labels as CSV files.
# Typical error: "Invalid label" means a label is not spelled exactly like the options.

# ------------------------------------------
# YOU MAY CHANGE THIS
# ------------------------------------------
GROUP_NAME = "group1"         # Your group name or number, no spaces!

# Options: Accel X, Accel Y, Accel Z, Gyro X, Gyro Y, Gyro Z, Unknown
LABEL_SIGNAL_1 = "Unknown"
LABEL_SIGNAL_2 = "Unknown"
LABEL_SIGNAL_3 = "Unknown"
LABEL_SIGNAL_4 = "Unknown"
LABEL_SIGNAL_5 = "Unknown"
LABEL_SIGNAL_6 = "Unknown"
# ------------------------------------------

VALID_LABELS = ["Accel X", "Accel Y", "Accel Z", "Gyro X", "Gyro Y", "Gyro Z", "Unknown"]
my_labels = [LABEL_SIGNAL_1, LABEL_SIGNAL_2, LABEL_SIGNAL_3,
             LABEL_SIGNAL_4, LABEL_SIGNAL_5, LABEL_SIGNAL_6]

for label in my_labels:
    if label not in VALID_LABELS:
        raise ValueError(f"Invalid label: '{label}'. Use one of: {VALID_LABELS}")

if " " in GROUP_NAME:
    raise ValueError("GROUP_NAME must not contain spaces.")

# ---- Data files ----
# REVIEW: Align format with Session 4.
#   - Column names: currently time_s, signal_1 ... signal_6
#   - Time column must be called 'time_s' (seconds)
#   - Decimal separator: currently '.', confirm (German Excel expects ',')
#   - Units: none on purpose in Session 3. Decide if Session 4 needs units
#     (e.g. g or m/s^2 for the accelerometer, deg/s for the gyroscope)
HEADER = ["time_s"] + [f"signal_{i}" for i in range(1, NUMBER_OF_SIGNALS + 1)]

saved_files = []
for name, (times, values) in RECORDINGS.items():
    filename = f"{GROUP_NAME}_{name}.csv"
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        for t, row in zip(times, values):
            writer.writerow([f"{t:.3f}"] + [f"{v:.5f}" for v in row])
    saved_files.append(filename)

if not RECORDINGS:
    print("No recordings found. Run Cell 2 and Cell 3 first.")

# ---- Labels file (separate, used by Session 4) ----
labels_filename = f"labels_{GROUP_NAME}.csv"
with open(labels_filename, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["signal", "label"])
    for i, label in enumerate(my_labels, start=1):
        writer.writerow([f"signal_{i}", label])
saved_files.append(labels_filename)

print("Saved files:")
for name in saved_files:
    print("  -", name)
print("DONE! Download these files from the file browser on the left (right-click > Download).")

# REVIEW: Explain where students must upload the files (their group folder, e.g. shared drive)
# so that Session 4 can find them. Add the exact folder/link here.

---

🗂️ [STRUCTURE]
**Wrap-up (end of session, no code):**

📝 [TEXT]
## Before you leave
- ✅ I downloaded all my CSV files (data and `labels_...csv`).
- ✅ I uploaded them to my group folder. REVIEW: add folder instructions.
- ✅ I wrote down which signals I am sure about and which I am not.

In Session 4, you will find out whether your guesses were right, and what these signals really measure.